# HMM Deployment and Trajectory Clustering Workflow Preview

This notebook renders the deployment-only HMM behavioral-state workflow from `behav3d.widgets.state_classification`, the timepoint-feature DTW workflow, a categorical DTW trajectory clustering workflow, and the behavioral trajectory classification workflow.

The HMM workflow covers:
- assign HMM intrinsic behavioral states
- combine / rename intrinsic clusters
- rename clusters assigned to binary groups
- create analysis plots
- apply a saved HMM deployment artifact
- open intrinsic or full backprojections

The feature-DTW workflow runs directly on extracted timepoint features and does not require HMM state classification. It can also generate original BEHAV3D-style track backprojection PDFs/MP4s from the Feature DTW clusters. The categorical trajectory workflow clusters whole behavioral-state trajectories from the production `FULL_STATE_COL` (`behavioral_state`) into method-specific output folders.


In [9]:
from pathlib import Path
import sys

import ipywidgets as widgets
from IPython.display import display

from behav3d.widgets.utils import PathPicker
from behav3d.widgets.metadata import MetadataLoader
from behav3d.widgets.state_classification import StateClassificationHMMDeploymentPanel
from behav3d.widgets.analysis import MotileCellAnalysisPanel
from behav3d.widgets.track_classification import TrackClassificationPanel
from behav3d.core.metadata import (
    detect_immune_cell_types_from_metadata,
    detect_organoid_types_from_metadata,
    detect_other_cell_types_from_metadata,
)



In [10]:
output_dir_picker = PathPicker(
    mode="dir",
    start_dir=".",
    description="Output Dir:",
)

metadata_path_picker = PathPicker(
    mode="file",
    start_dir=".",
    description="Metadata CSV:",
)
metadata_path_picker.filter_pattern = "*.csv"

metadata_loader = MetadataLoader(
    metadata_path_picker=metadata_path_picker,
    output_dir_picker=output_dir_picker,
)

render_btn = widgets.Button(
    description="Render HMM + trajectory workflows",
    button_style="success",
)
status_html = widgets.HTML("<i>Load metadata to render the HMM deployment and trajectory clustering panels.</i>")
panel_out = widgets.Output()


In [11]:
def _render_hmm_panel(_=None):
    panel_out.clear_output()
    with panel_out:
        if getattr(metadata_loader, "metadata", None) is None:
            status_html.value = "<b style='color:#a60;'>Load metadata first.</b>"
            return
        immune_types = detect_immune_cell_types_from_metadata(metadata_loader.metadata)
        organoid_types = detect_organoid_types_from_metadata(metadata_loader.metadata)
        other_types = detect_other_cell_types_from_metadata(metadata_loader.metadata)
        ordered_cell_types = []
        for cell_type in list(immune_types) + list(organoid_types) + list(other_types) + ["tcell"]:
            cell_type = str(cell_type).strip()
            if cell_type and cell_type not in ordered_cell_types:
                ordered_cell_types.append(cell_type)
        default_cell_type = ordered_cell_types[0]

        hmm_deployment_panel = StateClassificationHMMDeploymentPanel(metadata_loader=metadata_loader, cell_type=default_cell_type)
        feature_dtw_panel = MotileCellAnalysisPanel(metadata_loader=metadata_loader, cell_type=default_cell_type)
        dtai_trajectory_panel = TrackClassificationPanel(metadata_loader=metadata_loader, cell_type=default_cell_type)
        status_html.value = "<b style='color:#080;'>Rendered HMM deployment, feature-DTW, DTW, and behavioral trajectory classification panels.</b>"

        hmm_box = widgets.VBox(
            [
                widgets.HTML("<h4>HMM Deployment Artifact Workflow</h4>"),
                hmm_deployment_panel.ui,
            ],
            layout=widgets.Layout(gap="12px"),
        )
        feature_dtw_box = widgets.VBox(
            [
                widgets.HTML("<h4>Timepoint Feature DTW Analysis</h4>"),
                widgets.HTML(
                    "<span style='color:#555;'>Runs <code>behav3d.analysis.tcell_analysis.run_tcell_analysis</code> "
                    "on extracted timepoint features, then fits UMAP and K-means clusters. "
                    "Use the track backprojection controls below the run button to export original BEHAV3D-style cluster tracks.</span>"
                ),
                feature_dtw_panel.ui,
            ],
            layout=widgets.Layout(gap="12px"),
        )
        dtai_box = widgets.VBox(
            [
                widgets.HTML("<h4>Behavioral trajectory classification</h4>"),
                dtai_trajectory_panel.ui,
            ],
            layout=widgets.Layout(gap="12px"),
        )
        workflow_tabs = widgets.Tab(children=[hmm_box, feature_dtw_box, dtai_box])
        workflow_tabs.set_title(0, "HMM deployment")
        workflow_tabs.set_title(1, "Feature DTW analysis")
        workflow_tabs.set_title(2, "Behavioral trajectory classification")
        workflow_tabs.selected_index = 1

        display(
            widgets.VBox(
                [
                    widgets.HTML(
                        "<b>Workflow panels:</b> use the tabs below to switch between "
                        "HMM deployment, timepoint-feature DTW, and behavioral trajectory classification. "
                        "The feature-DTW tab is selected by default because it does not require HMM states."
                    ),
                    workflow_tabs,
                ],
                layout=widgets.Layout(gap="12px"),
            )
        )


_original_load = metadata_loader.load

def _wrapped_load(*args, **kwargs):
    result = _original_load(*args, **kwargs)
    if getattr(metadata_loader, "metadata", None) is not None:
        status_html.value = "<b style='color:#080;'>Metadata loaded. Auto-rendering HMM, feature-DTW, and behavioral trajectory classification panels...</b>"
        _render_hmm_panel()
    return result


metadata_loader.load = _wrapped_load
render_btn.on_click(_render_hmm_panel)

display(widgets.VBox([
    widgets.HTML("<h3>Render HMM Deployment and Trajectory Clustering Workflows</h3>"),
    output_dir_picker,
    metadata_path_picker,
    metadata_loader.button,
    metadata_loader.out,
    render_btn,
    status_html,
    panel_out,
]))


## Notes

- The HMM panel remains the production HMM deployment workflow.
- The feature-DTW tab reuses `behav3d.widgets.analysis.MotileCellAnalysisPanel`, writes `run_tcell_analysis` outputs to `analysis/<cell_type>/timepoint_feature_dtw/`, and can export cluster track backprojections under `analysis/<cell_type>/timepoint_feature_dtw/clustering/example_tracks/backprojection/`.
- The categorical state DTW panel lives in `test/trajectory_dtw_classification.py` and writes to `analysis/<cell_type>/behavioral_state_trajectories_dtw/`.
- The behavioral trajectory classification panel reuses `behav3d.widgets.track_classification.TrackClassificationPanel` and writes to `analysis/<cell_type>/behavorial_trajectories/`.
- The preview renders HMM deployment, feature-DTW analysis, categorical state DTW clustering, and behavioral trajectory classification as separate tabs.
- Categorical state DTW always uses `FULL_STATE_COL` from `behav3d.analysis.clustering.state.classification`, currently `behavioral_state`; this column is fixed and not user-selectable.
- Categorical state DTW uses `dtwParallel` with `local_dissimilarity=hamming` for categorical sequences, and records the selected backend settings in `.uns`.
- Behavioral trajectory classification uses one-hot encoded categorical states with `dtaidistance.dtw_ndim.distance_matrix_fast` and separates clustering from plot creation in its widget.
- DTW example cluster PDFs are written under `analysis/<cell_type>/behavioral_state_trajectories_dtw/clustering/example_tracks/`.
- DTAI example cluster PDFs are written under `analysis/<cell_type>/behavorial_trajectories/clustering/example_tracks/`.
- Only trajectory size, number of clusters, seed, and Advanced are shown by default; linkage, parallel execution, save distance matrix, original BEHAV3D mode, and plot/backprojection controls remain available in the workflow.
